## Find features in proximity to sensors

Find 

### Code initialisation

In [2]:
import pandas as pd
import geopandas as gpd
from IPython.display import Image
from shapely.geometry import Point, Polygon
import folium
import numpy as np
from shapely.ops import transform
import pyproj

from shapely.errors import ShapelyDeprecationWarning
import warnings
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 

buffer_size= 100

### Functions

In [207]:
def draw_buffers(sensors_gdf):
    # Convert CRS of sensor data to one that works with finding the radius
    reprojected_sensors = sensors_gdf.copy()
    reprojected_sensors.to_crs(crs=3577, inplace = True)

    # Draw a buffer around each sensor in metres
    reprojected_sensors['polygon'] = reprojected_sensors.geometry.buffer(buffer_size_m)   

    return reprojected_sensors

# Function which finds which features from a geodataframe of features are within the radius of a sensor
# and saves the location and feature sub-types of these
def find_features_near_sensor (sensor_num, gdf, buffer_size_m, grouping_col_name):
    # Convert CRS of sensor data to one that works with finding the radius
    global buffered_sensors

    # Get geodataframe from just this sensor
    this_sensor_gdf = buffered_sensors[buffered_sensors.sensor_id == sensor_num]
    this_sensor_gdf.reset_index(inplace = True, drop = True)

    # Set up reprojection
    project = pyproj.Transformer.from_crs(pyproj.CRS('EPSG:4326'), pyproj.CRS('EPSG:3577'), always_xy=True).transform

    # Check if each of the features is within the radius of this sensor
    # Save the locations and themes of these features into two lists
    locations_in_radius = []
    themes_in_radius = []
    for feature in gdf.iterrows():
        transformed_point = transform(project, feature[1].geometry)
        if this_sensor_gdf['polygon'][0].contains(transformed_point):
            locations_in_radius.append(feature[1].geometry)
            themes_in_radius.append(feature[1][grouping_col_name])
            
    return locations_in_radius, themes_in_radius

In [253]:
# buffered_sensors = draw_buffers(sensors_gdf)
# buffered_sensors

reprojected_sensors = sensors_gdf.copy()
reprojected_sensors.to_crs(crs=3577, inplace = True)
reprojected_sensors

project = pyproj.Transformer.from_crs(pyproj.CRS('EPSG:3577'), pyproj.CRS('EPSG:4326'), always_xy=True).transform
reprojected_sensors['polygon'] = transform(project, this_sensor['polygon'][0])

,sensor_id,Latitude,Longitude,geometry
0,16,-37.815734,144.965210,POINT (1146630.553 -4192373.880)
1,50,-37.798082,144.967210,POINT (1147004.120 -4190454.435)
2,73,-37.816957,144.954154,POINT (1145642.411 -4192408.870)
3,66,-37.810578,144.964443,POINT (1146620.528 -4191801.142)
4,59,-37.808256,144.963049,POINT (1146523.582 -4191533.787)
...,...,...,...,...
86,51,-37.808418,144.959063,POINT (1146170.495 -4191515.769)
87,63,-37.813331,144.966756,POINT (1146793.613 -4192123.976)
88,87,-37.804549,144.949219,POINT (1145345.955 -4191002.878)
89,52,-37.812522,144.961940,POINT (1146378.220 -4191991.942)


# **Read in data**
## <ins>Sensors</ins>

In [4]:
sensors_gdf = create_features_gdf('Data/melbourne_locations.csv')

### Draw a circle around each sensor, within which to look for features

In [139]:
# Filter out just the columns we want
sensors_gdf = sensors_gdf[['sensor_id', 'Latitude', 'Longitude', 'geometry']]

## <ins>Feature data</ins>

In [6]:
buildings_gdf = create_features_gdf('Cleaned_data/buildings_clean.csv')
lights_gdf = create_features_gdf('Cleaned_data/lights_clean.csv')
furniture_gdf = create_features_gdf('Cleaned_data/street_inf_clean.csv')
bikes_gdf = create_features_gdf('Cleaned_data/bikes_clean.csv')
landmarks_gdf = create_features_gdf('Cleaned_data/landmarks_clean.csv')

# Find number of each feature within the radius of each sensor

NB: If you use the full buildings_gdf with this method then it returns way too many buildings as each building is listed seperately in each year  
Can check this with:  
buildings_gdf[buildings_gdf.duplicated(['Longitude', 'Latitude'], keep=False)].sort_values(['Longitude', 'Latitude'])[0:12]

In [18]:
buffer_size_m = 100
fp_to_save_to =  "Cleaned_data/features_near_sensors_{}.csv".format(buffer_size_m)

In [7]:
# Create a dictionary containing all the feature gdfs, and their names
gdfs = {'bikes': bikes_gdf, 'lights':lights_gdf, 'furniture': furniture_gdf, 'landmarks':landmarks_gdf}

# Add a gdf containing just the buildings present in each year to the dictionary of gdfs
for year in buildings_gdf.year.unique():
    this_year = buildings_gdf[buildings_gdf.year == year].copy()
    this_year.reset_index(inplace=True, drop = True)
    gdfs["buildings_{}".format(year)] = this_year    

In [140]:
if not os.path.isfile(fp_to_save_to):


    # Save
    num_features_near_sensor.to_csv(fp_to_save_to, index = False)
else:
    num_features_near_sensor = pd.read_csv(fp_to_save_to)

In [192]:
# List of gdfs, themes and subthemes
gdfs = [furniture_gdf, landmarks_gdf, bikes_gdf, lights_gdf]
gdf_types= ['furniture', 'landmarks', 'bikes', 'lights']
gdf_subtheme_column_name = ['feature', 'theme', 'capacity', "lamp_rating_w"]

# For counting the number of features:
# Keys will be sensor numbers, items will be list of number of each type of feature in this sensor's radius
num_features = {}
# Keys will be sensor numbers, items will be a dataframe containing location and type of features in sensor radius
feature_details = {}

for sensor_num in sorted(sensors_gdf['sensor_id'].unique().tolist()):
    # Lists to store the number of features for each gdf for this sensor
    n_features_this_sensor = []
    
    # Dataframe to contain location and feature types from all gdfs, for this sensor 
    features_df_this_sensor = pd.DataFrame(None)
    
    # loop through each geodataframe
    for i in range(0,len(gdfs)):
        # Find the features from this geodataframe in the radius of this sensor
        features_this_sensor = find_features_near_sensor(sensor_num, gdfs[i], buffer_size_m, gdf_subtheme_column_name[i])
        
        # Create dataframe with the location and type of feature for this geodataframe nr the sensor
        features_df_this_gdf= pd.DataFrame({'Location': features_this_sensor[0], 'Type': gdf_types[i], 
                                                 'Subtype': features_this_sensor[1]})
        # Add to dataframe containing results from all geodataframes
        features_df_this_sensor = pd.concat([features_df_this_sensor,features_df_this_gdf],axis=0)
        
        # Add the number of features of this type to the list
        n_features_this_sensor.append(len(features_df_this_gdf))
    
    # Add number/feature details to dictionaries
    num_features[sensor_num] = n_features_this_sensor
    feature_details[sensor_num] = features_df_this_sensor

In [203]:
test = pd.DataFrame(num_features).T
test.columns = gdf_types
test

,furniture,landmarks,bikes,lights
1,256,2,0,56
2,257,2,0,34
3,153,1,1,28
4,193,2,0,0
5,138,0,0,23
...,...,...,...,...
88,135,0,1,4
89,83,1,0,0
90,111,0,0,0
91,46,0,0,2


In [204]:
feature_details[92]
values, counts = np.unique(feature_details[1]['Type'], return_counts=True)
df = pd.DataFrame({"Values": values, 'Counts': counts})
df

,Values,Counts
0,furniture,256
1,landmarks,2
2,lights,56


In [66]:
# feature_types = test(1, furniture_gdf, 'feature' )
# ttt = feature_types[1]
values, counts = np.unique(ttt, return_counts=True)
df = pd.DataFrame({"Values": values, 'Counts': counts})
df
# features_in_radius_by_type_df = features_in_radius_df.groupby(grouping_col_name).agg({'geometry': 'count'})
# # Add theme as a column        
# features_in_radius_by_type_df.insert(0, grouping_col_name, features_in_radius_by_type_df.index)
# # Reset index (so theme is no longer the index)
# features_in_radius_by_type_df.reset_index(inplace=True, drop = True)
# features_in_radius_by_type_df.rename(columns={"geometry": sensor_num}, inplace = True)   

,Values,Counts
0,Bicycle Rails,31
1,Bollard,81
2,Drinking Fountain,4
3,Floral Crate/Planter Box,20
4,Information Pillar,2
5,Litter Bin,26
6,Seat,92


# Plot the features within the radius for one sensor

### Get the feature gdfs that you wish to check for in the sensor's radius

In [22]:
# dictfilt = lambda x, y: dict([ (i,x[i]) for i in x if i in set(y) ])
# filtered_gdfs = dictfilt(gdfs, ("bikes","lights", "landmarks", "furniture", "buildings_2011"))

## Plot

In [240]:
this_sensor = buffered_sensors[buffered_sensors['sensor_id'] ==sensor_num].copy()

In [398]:
this_sensors_features = feature_details[2]
this_sensors_features['Longitude'] = this_sensors_features.Location.apply(lambda p: p.x)
this_sensors_features['Latitude'] = this_sensors_features.Location.apply(lambda p: p.y)

# Plot
melbourne_map = folium.Map(location=[this_sensor.Latitude.mean(),
                           this_sensor.Longitude.mean()], zoom_start=18, control_scale=True)#, min_zoom = 13)
# Add sensor location marker
folium.Marker([this_sensor["Latitude"], this_sensor["Longitude"]], popup=this_sensor["sensor_id"], 
              icon = folium.Icon(color="red")).add_to(melbourne_map)

# Add buffer zone
folium.GeoJson(data=this_sensor["polygon"]).add_to(melbourne_map)

# # Add features
for index in range(0, len(this_sensors_features)):
    feature = this_sensors_features[this_sensors_features.index ==index].copy()
    folium.CircleMarker([feature['Latitude'], feature['Longitude']], popup=feature["Type"],
        color = 'black', fill_color = 'white', fill = 'True', fill_opacity = True).add_to(melbourne_map)
       
melbourne_map

In [39]:
# Some kind of category to denote a really tall building (not sure if an advantage to making it 
# categorical rather than continuous though)
building_gdf_2011 = gdfs["buildings_{}".format(2011)]
building_gdf_2011['HeightOfBuilding']= pd.cut(building_gdf_2011['n_floors'], bins=[0,3,20, 50, float('Inf')], labels=['small', 'medium', 'large', 'MASSIVE'])

# Create dataframes containing the number of sub-categories of features 

In [193]:
# landmarks = feature_types(landmarks_gdf, 'theme')
# furniture = feature_types(furniture_gdf, 'feature')
# buildings_2012 = feature_types(gdfs['buildings_2012'], 'building_use')

## Find the buildings within the 'proximity' of each sensor, in a particular year

In [ ]:
def bike_locations(sensor_num):
    gdf = sensors_gdf[sensors_gdf.sensor_id == sensor_num]
    sen_bikes = []
    for point in bike_gdf.geometry:
        for geo in gdf['polygon']:
            if geo.contains(point):
                sen_bikes.append(list(point.coords))
    bike_loc = [x[0] for x in sen_bikes]
    sensor_bikes = pd.DataFrame({'latitude': [x[0] for x in bike_loc], 'longitude': [x[1] for x in bike_loc]})
    return sensor_bikes

In [ ]:
bike_locations(3)

### create dataframe of all location points connected with sensor id

In [ ]:
sensor_bike_points = pd.DataFrame([])
for x in sensors_gdf.sensor_id.unique():
    sensor_points = bike_locations(x)
    sensor_points['sensor_id'] = x
    sensor_bike_points = sensor_bike_points.append(sensor_points)

In [ ]:
sensor_bike_points.to_csv('./location features/features with locations/bike_points.csv', header = sensor_bike_points.columns, index=False)

### create dataframe of all location points connected with sensor id

In [ ]:
sensor_light_points = pd.DataFrame([])
for x in sensors_gdf.sensor_id.unique():
    sensor_points = lights_locations(x)
    sensor_points['sensor_id'] = x
    sensor_light_points = sensor_light_points.append(sensor_points)

In [ ]:
sensor_light_points.to_csv('./location features/features with locations/light_points.csv', header = sensor_light_points.columns, index=False)

### List the number of lights in a sensor's radius

In [ ]:
light_dict = {}
keys = sensors.sensor_id.unique()
for k in keys:
    light_dict[k] = len(lights_locations(k))


In [ ]:
#convert dictionary to dataframe
lighting_df = pd.DataFrame(light_dict.items(), columns=['sensor_id', 'num_lights'])
lighting_df

In [ ]:
def lights_in_radius(sensor_num):
    return lighting_df[lighting_df.sensor_id == sensor_num]

In [ ]:
lights_in_radius(2)

# Landmarks

In [ ]:
 #zip lists of theme and point
landmark_zip_list =  list(zip(landmarks_gdf.theme,landmarks_gdf.geometry))
landmark_zip_list[:10]

### returns data frame with location coordinates and theme of landmarks in a sensor's radius

In [ ]:
def landmarks_coords(sensor_num):
    landmark_coord = []
    theme = []
    gdf = sensors_gdf[sensors_gdf.sensor_id == sensor_num]
    for x in landmark_zip_list:
        for geo in gdf['polygon']:
            if geo.contains(x[1]):
                landmark_coord.append(x[1])
                theme.append(x[0])
    zipped_sensor = list(zip(landmark_coord, theme))
    landmark = pd.DataFrame(zipped_sensor, columns = ['point', 'theme'])
    landmark['longitude'] = landmark.point.apply(lambda p: p.x)
    landmark['latitude'] = landmark.point.apply(lambda p: p.y)
    return landmark

In [ ]:
landmarks_coords(2)

### create dataframe of all location points connected with sensor id

In [ ]:
sensor_landmark_points = pd.DataFrame([])
for x in sensors_gdf.sensor_id.unique():
    sensor_points = landmarks_coords(x)
    sensor_points['sensor_id'] = x
    sensor_landmark_points = sensor_landmark_points.append(sensor_points)

In [ ]:
sensor_landmark_points.to_csv('./location features/features with locations/landmark_points.csv', header = sensor_landmark_points.columns, index=False)

### returns number of different themes of landmarks in a sensor's radius

In [ ]:
def landmarks_in_radius(sensor_num):
    landmark_coord = []
    theme = []
    gdf = sensors_gdf[sensors_gdf.sensor_id == sensor_num]
    for x in landmark_zip_list:
        for geo in gdf['polygon']:
            if geo.contains(x[1]):
                landmark_coord.append(x[1])
                theme.append(x[0])
                
    if len(landmark_coord) > 0: 
        dicts = {}
        for x in landmarks.theme.unique():
            dicts[x] = 0
           
        zipped_sensor = list(zip(landmark_coord, theme))
        df_zip = pd.DataFrame(zipped_sensor, columns = ['point', 'theme'])
        group = df_zip.groupby('theme').agg({'point': 'count'})
        transform_group = group.T
        
        for col in transform_group.columns:
            dicts[col] = transform_group[col][0]
            
        df = pd.DataFrame(dicts.items())
        df.set_index(df.columns[0], inplace = True)
        new_df = df.T
        new_df['sensor_id'] = sensor_num
        new_df.set_index('sensor_id', inplace = True)
        return new_df
    
    else:
        dicts = {}
        for x in landmarks.theme.unique():
            dicts[x] = 0
        df = pd.DataFrame(dicts.items())
        df.set_index(df.columns[0], inplace = True)
        new_df = df.T
        new_df['sensor_id'] = sensor_num
        new_df.set_index('sensor_id', inplace = True)
        return new_df

In [ ]:
landmarks_in_radius(2)

In [85]:
#create list of type of features and point
furniture_zip_list =  list(zip(furniture_gdf['feature'],furniture_gdf['geometry']))
furniture_zip_list[:10]

[('Bicycle Rails', <shapely.geometry.point.Point at 0x7faeda952fd0>),
 ('Bicycle Rails', <shapely.geometry.point.Point at 0x7faed7d5d3d0>),
 ('Bicycle Rails', <shapely.geometry.point.Point at 0x7faed5fc57d0>),
 ('Bollard', <shapely.geometry.point.Point at 0x7faed7d5dc10>),
 ('Bicycle Rails', <shapely.geometry.point.Point at 0x7faed7d5dbd0>),
 ('Bicycle Rails', <shapely.geometry.point.Point at 0x7faeda952c50>),
 ('Bicycle Rails', <shapely.geometry.point.Point at 0x7faed7d5db90>),
 ('Bicycle Rails', <shapely.geometry.point.Point at 0x7faed7d5dc90>),
 ('Bicycle Rails', <shapely.geometry.point.Point at 0x7faed7d5d050>),
 ('Litter Bin', <shapely.geometry.point.Point at 0x7faeda952a90>)]

In [ ]:
def furniture_coords(sensor_num):
    furniture_coord = []
    feature = []
    gdf = sensors_gdf[sensors_gdf.sensor_id == sensor_num]
    for x in furniture_zip_list:
        for geo in gdf['polygon']:
            if geo.contains(x[1]):
                furniture_coord.append(x[1])
                feature.append(x[0])
                
    zipped_sensor = list(zip(furniture_coord, feature))
    furniture = pd.DataFrame(zipped_sensor, columns = ['point', 'feature'])
    furniture['longitude'] = furniture.point.apply(lambda p: p.x)
    furniture['latitude'] = furniture.point.apply(lambda p: p.y)
    return furniture

In [ ]:
furniture_coords(2)

### create dataframe of all location points connected with sensor id

In [ ]:
sensor_furniture_points = pd.DataFrame([])
for x in sensors_gdf.sensor_id.unique():
    sensor_points = furniture_coords(x)
    sensor_points['sensor_id'] = x
    sensor_furniture_points = sensor_furniture_points.append(sensor_points)

In [ ]:
sensor_furniture_points.to_csv('./location features/features with locations/furniture_points.csv', header = sensor_furniture_points.columns, index=False)

### find number of features in sensor's radius, if there is none of a type, return 0

In [ ]:
def furniture_in_radius(sensor_num):
    furniture_coord = []
    feature = []
    gdf = sensors_gdf[sensors_gdf.sensor_id == sensor_num]
    for x in furniture_zip_list:
        for geo in gdf['polygon']:
            if geo.contains(x[1]):
                furniture_coord.append(x[1])
                feature.append(x[0])
                
    if len(furniture_coord) > 0: 
        dicts = {}
        for x in furniture_gdf['feature'].unique():
            dicts[x] = 0
           
        zipped_sensor = list(zip(furniture_coord, feature))
        df_zip = pd.DataFrame(zipped_sensor, columns = ['point', 'feature'])
        group = df_zip.groupby('feature').agg({'point': 'count'})
        transform_group = group.T
        
        for col in transform_group.columns:
            dicts[col] = transform_group[col][0]
            
        df = pd.DataFrame(dicts.items())
        df.set_index(df.columns[0], inplace = True)
        new_df = df.T
        new_df['sensor_id'] = sensor_num
        new_df.set_index('sensor_id', inplace = True)
        return new_df
    
    else:
        dicts = {}
        for x in furniture_gdf['feature'].unique():
            dicts[x] = 0
        df = pd.DataFrame(dicts.items())
        df.set_index(df.columns[0], inplace = True)
        new_df = df.T
        new_df['sensor_id'] = sensor_num
        new_df.set_index('sensor_id', inplace = True)
        return new_df

In [ ]:
furniture_in_radius(2)

# Create a dataframe with all the features in a sensor's radius, per year

In [86]:
def sensor_features_per_year(sensor_num, year):
    return pd.concat([furniture_in_radius(sensor_num), landmarks_in_radius(sensor_num),  buildings_in_radius(sensor_num, buildings_per_year(year)),
           bike_df.loc[[sensor_num]], lighting_df.loc[[sensor_num]]], axis = 1)



In [87]:
sensor_features_per_year(2, 2012)

NameError: name 'furniture_in_radius' is not defined

## Make full dataframe per year, with all sensors that are valid for that period
yearly list of valid sensors is pulled from the data cleaning notebook

In [88]:
sensors_2011 = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
sensors_2012 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
sensors_2013 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 16, 17, 18]
sensors_2014 = [2,3,4,5,6,8,9,10,11,12,13,14,15,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32]
sensors_2015 = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 18, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 36, 38, 39, 40]
sensors_2016 = [1, 2, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 36, 37, 38, 39, 40, 42, 43, 44, 53]
sensors_2017 = [1, 2, 3, 6, 7, 8, 9, 10, 11, 12, 15, 17, 18, 19, 20, 21, 23, 24, 25, 26, 27, 28, 29, 31, 33, 34, 35, 37, 39, 40, 42, 43, 44, 53]
sensors_2018 = [1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 15, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 40, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53]
sensors_2019 = [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 14, 15, 17, 18, 19, 20, 21, 23, 24, 26, 27, 28, 30, 31, 34, 35, 36, 37, 39, 40, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 56, 57, 58]

In [89]:
def features_per_year(year, year_list):
    year_df = pd.DataFrame([])
    for x in year_list:
        year_df = year_df.append(sensor_features_per_year(x, year))
        
    df = year_df.loc[:,~year_df.columns.duplicated()]
    #save to csv
    df.to_csv('./location features/features_{}.csv'.format(year), header = df.columns, index=True)
    
    

In [90]:
features_per_year(2011, sensors_2011)
features_per_year(2012, sensors_2012)
features_per_year(2013, sensors_2013)
features_per_year(2014, sensors_2014)
features_per_year(2015, sensors_2015)
features_per_year(2016, sensors_2016)
features_per_year(2017, sensors_2017)
features_per_year(2018, sensors_2018)

NameError: name 'furniture_in_radius' is not defined